# TICKLE Phase 1
## Can 10 GHz rain-scatter propagation be labeled from open data?

**Michelle Thompson W5NYV**
September 2026
[github.com/Abraxas3d/TICKLE](https://github.com/Abraxas3d/TICKLE)

TICKLE aims to predict when 10 GHz "cloud bounce" (rain scatter) will work
between Southern California and Arizona, eventually using machine learning on
GOES-18 satellite imagery. This document records **Phase 1**: no model is
trained here. Instead we answer the question every observational-ML project
must answer first. *where will trustworthy labels come from?* We validate
the answer against seven years of amateur radio contest logs, archived NEXRAD
radar, and terrain geometry.

**Phase 1 findings, in brief:**
1. The community folklore ("San Diego to Arizona works during August/September
   monsoon storms") is physically grounded and statistically visible in
   contest logs, surviving an operator-activity control.
2. Archived NEXRAD Level II radar can generate propagation labels from
   physics alone. This was validated against a real 2021 rain-scatter session.
3. **Naive radar labeling over-estimates opportunity by up to three orders of
   magnitude.** Terrain-derived common-volume geometry is a prerequisite, not
   a refinement. On our benchmark day it cut one radar's apparent scattering
   volume by ~100–3000× while *relocating* the day's workable mirrors to the
   western corridor.
4. You do not have to be on a mountaintop for cloud bounce. A home station
   in Del Mar, CA (with a modest mast) is geometrically viable for the
   Parker, AZ path. Common-volume floor ≈ 2.0 km MSL, far below monsoon
   cell tops.

---
### What this document is and what it is not

**This is a feasibility study and research prospectus, written as a literate
program.** It contains no trained model, no inference, and no forecast. Its
purpose is to establish, with evidence a reader can re-execute, that the
prediction problem is well-stated and that its prerequisites exist.

**Claims established here (each validated in the section cited):**

- **C1.** The SoCal↔AZ 10 GHz corridor is real and convectively modulated.
  Corridor QSO timing tracks the monsoon diurnal cycle after controlling for
  operator activity (§3).
- **C2.** Propagation labels can be manufactured from open archived radar
  with no on-air measurements, and they agree in detail with a real
  radar-era QSO session, including its onset, its distance-vs-mirror-size
  threshold behavior, and its end (§4).
- **C3.** Terrain-derived common-volume geometry is a *prerequisite* for
  honest labels, not a refinement. Omitting it inflates apparent opportunity
  up to on the order of 10^3 and can misplace which storms matter (§5).
- **C4.** The operational home path (Del Mar, with a mast) is geometrically
  viable, and the region where its usable mirrors form is mapped (§5).

**Claims deliberately NOT made:** that any quantity has been *predicted*.
All analysis here is retrospective and concurrent. This is diagnosis, not 
forecast. Whether GOES imagery carries predictive signal with useful lead 
time, and whether any learned model beats a bistatic-physics baseline built 
on these same floor maps, are the falsifiable questions of Phase 2. Here we
strive to identify useful labels. 

---

Everything below reads from files produced by the pipeline scripts in this
repository. Paths are absolute to this machine's layout. Modify them for yours.

In [ ]:
import csv, sqlite3, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

TICKLE = "/Users/w5nyv/TICKLE"
DATA   = os.path.join(TICKLE, "data")

def show(path, caption=""):
    if os.path.exists(path):
        if caption: print(caption)
        display(Image(filename=path))
    else:
        print(f"(figure not found: {path} -- run the producing script first)")

## 1. The question and the physics

10 GHz (3 cm) is the amateur "Rainy Day Band". Scattering efficiency from
hydrometeors rises steeply with drop size up to roughly a tenth of a
wavelength, making thunderstorm rain shafts effective bistatic reflectors at
this frequency. Two stations that cannot see each other can both see a storm
cell. The cell becomes the mirror. North American Monsoon convection over the
Arizona/California deserts (July–September) produces the region's largest
hydrometeors and tallest cells. Hence the folklore that the SD to AZ path opens
on monsoon afternoons.

The word-of-mouth version came with real uncertainty. Which endpoints, what
distances, how often, and is the timing atmospheric or just when operators
happen to be on mountains? Phase 1 follows folklore with measurements.

## 2. Data assets

**Backyard GOES-18 HRIT archive.** 646,242 files / 242.7 GB captured in Del
Mar: goestools-decoded JPGs (bands C02, C07, C13 + false color) from
mesoscale sectors M1/M2 at ~15-minute cadence, August 2022 – May 2025 core
(~480k usable frames), with a 5-month station outage after May 2025.
A SQLite manifest (`Step_1/build_manifest.py`) indexes every file from
filename metadata alone. GOES-18 data can also be obtained online. 

**Sector pointing, recovered from the pictures themselves.** Mesoscale
sectors move at NOAA's discretion, and HRIT JPGs carry no navigation. But,
goestools burned map overlays into every frame. `Step_1p5/estimate_pointing.py`
extracts the overlay lines and chamfer-matches them against Natural Earth
coastlines/borders projected through the GOES fixed-grid, recovering each
dwell's pointing to +/- 5 km (validated on frames with known geography).

**Verdict:** NOAA rarely watched our corridor. M2 spent 77% of its life on
the Gulf of Alaska, M1 58% on the Sierra. Only ~2.4% (M1) and ~0.5% (M2) of
frames cover SD to AZ corridor. 

**Consequence:** model training imagery will come from
NOAA's full CONUS archive on AWS (5-min cadence, calibrated L1b, always
covers the corridor). The backyard archive's roles are pipeline validation
and internet-independent live inference. Silver lining: four multi-hour
corridor dwells landed in monsoon season, including a continuous 24 h stare
on 2023-09-01 and are used as the benchmark day in §5.

In [ ]:
# Corridor-covering dwells from the pointing census
m = sqlite3.connect(os.path.join(DATA, "goes18_manifest.db"))
try:
    dw = pd.read_sql_query(
        "SELECT product, seg_start, seg_end, n_frames FROM pointing_segments "
        "WHERE covers_corridor=1 ORDER BY seg_start", m)
    print(f"{dw.n_frames.sum():,} corridor-covering frames in {len(dw)} dwells")
    dw
except Exception as e:
    print("pointing_segments unavailable:", e)
dw

### Key dates registry

Every date that carries weight in this project, and why. When extending the
work, start here before picking a new replay target.

| Date (UTC) | Why it matters | Status |
|---|---|---|
| **2021-09-18** | Parker rover session: AZ rover at DM23XQ works a 5-hour chain of SoCal stations at 293–393 km during the 10 GHz and Up contest. **Radar-validated as rain scatter** (§4) this is the event that proved the label generator. *Predates the GOES archive* (starts Aug 2022), so radar-only. | Replayed, sanmiguel_parker |
| **2023-09-01** | **The benchmark day.** M2 mesoscale sector stared at the corridor continuously for ~24 h *during peak monsoon* Every radar volume has a backyard GOES frame within +/- 8 min. Source of the flat-proxy vs. terrain-floor finding (§5) and the first GOES-to-radar training pairs. | Replayed both ways (flat proxy, then terrain floor), sanmiguel_parker |
| 2022-08-10→11 | Corridor-covering M2 dwell, 22 h, monsoon season. Untested pilot candidate. Same triple-source potential as the benchmark day. | Not yet replayed |
| 2022-08-24→25 | Corridor-covering M2 dwell, 24 h, monsoon season. Same. | Not yet replayed |
| 2024-08-08→09 | Corridor-covering M1 dwell, ~21 h, monsoon season. Same. | Not yet replayed |
| 2025-05-14 | Backyard GOES station goes dark and there is only a trickle of frames after Oct 2025. End of the archive's solid core. | — |
| Contest weekends | 3rd full weekends of Aug and Sep, every year 2019–2025: where all 90 corridor QSOs live. Bulk label generation targets these first. | 2 of ~28 days replayed |
| **2026-09-19 to 2026-09-21** | ARRL 10 GHz & Up Leg 2, the next live data-collection window: monitoring, logging, and AZ beacon-host recruiting. | Upcoming |

Zero overlap exists between corridor-covering GOES dwells and corridor
contest QSOs (verified by cross-query). So, the triple coincidence never
happened, which is why the benchmark day (GOES+radar) and the Parker day
(radar+QSOs) are separate pillars.

## 3. Ground truth: seven years of ARRL 10 GHz & Up logs

The ARRL log portal supplied all retained Cabrillo logs (2019–2025): **992
logs, 70,636 QSO lines, 100% parsed** after accommodating one operator's
8-character extended grid squares. Each QSO line is a timestamped,
grid-located success record. 90 unique QSOs cross the SoCal to AZ-desert
corridor.

The key epistemic hazard: contest QSOs cluster when operators are active, so
raw timing proves nothing about propagation. The control below compares
corridor QSOs against short-range (<100 km) SoCal QSOs. These are contacts 
that need no scatter and therefore trace pure operator behavior.

In [ ]:
q = sqlite3.connect(os.path.join(DATA, "contest_qsos.db"))
df = pd.read_sql_query(
    """SELECT * FROM qsos WHERE band_ghz BETWEEN 9.5 AND 11
       AND lat1 IS NOT NULL AND lat2 IS NOT NULL""", q, parse_dates=["ts"])

def in_socal(la, lo): return 32.0 <= la <= 34.5 and -118.5 <= lo <= -115.8
def in_azdes(la, lo): return 31.5 <= la <= 35.5 and -115.8 <= lo <= -110.5
s1 = df.apply(lambda r: in_socal(r.lat1, r.lon1), axis=1)
s2 = df.apply(lambda r: in_socal(r.lat2, r.lon2), axis=1)
a1 = df.apply(lambda r: in_azdes(r.lat1, r.lon1), axis=1)
a2 = df.apply(lambda r: in_azdes(r.lat2, r.lon2), axis=1)
df["pair_key"] = df.apply(lambda r: (r.ts.isoformat(),)
                          + tuple(sorted([r.call1, r.call2])), axis=1)
corridor = df[(s1 & a2) | (s2 & a1)].drop_duplicates("pair_key")
control  = df[(df.dist_km < 100) & s1].drop_duplicates("pair_key")

hours = list(range(15, 24)) + list(range(0, 9))
def share(d):
    c = d.ts.dt.hour.value_counts()
    return [100 * c.get(h, 0) / len(d) for h in hours]

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(share(corridor), "o-", label=f"Corridor SD↔AZ (n={len(corridor)})")
ax.plot(share(control), "s--", label=f"Control <100 km (n={len(control):,})")
ax.axvspan(hours.index(20), hours.index(3), alpha=0.12,
           label="convective window 20z–03z")
ax.set_xticks(range(len(hours)))
ax.set_xticklabels([f"{h:02d}z" for h in hours])
ax.set_ylabel("% of group's QSOs"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Corridor vs. operator-activity control, by UTC hour")
plt.tight_layout(); plt.show()

**Reading the control:** the corridor *under*-performs at the control's
activity peak (19z ≈ local noon, pre-convective — maximum effort, minimum
result) and *over*-performs at 20z–21z (convective initiation) and 01z–03z
(mature evening storms persisting after casual operators quit). An
activity-only explanation cannot produce a deficit at the activity peak; a
convective explanation requires one. Caveats: n=90, QSOs cluster in sessions
(effective sample smaller), no formal significance test. But, the signature
matches monsoon physics, and §4 confirms it per-event with radar.

## 4. Radar replay: labels from physics, validated against real QSOs

`Step_4/nexrad_replay.py` pulls archived NEXRAD Level II volumes from the
AWS open archive (`unidata-nexrad-level2` this is the bucket that replaced
`noaa-nexrad-level2` in mid-2025), extracts reflectivity inside the path's
forward-scatter ellipse and common-volume height band, and emits a timeline
of scattering-relevant metrics per volume scan.

**Metric definitions and their honesty labels:**
- `max_dbz` — hottest gate in the corridor volume. dBZ = 10*log10(Z), Z the
  D^6-weighted drop population (mm^6/m^3). This is the same population that
  scatters a 10 GHz signal, so it is monotonically the right physics, though
  not yet a link budget. Vulnerable to single-gate clutter/AP spikes.
- `area40` — count of gates ≥40 dBZ. **A gate count, not km^2**: gate
  cross-section grows with range, so this ranks moments within a day but is
  not yet comparable across ranges/radars. v2: volume-weighted km^3 and a
  high-percentile dBZ statistic.

**Test case: the 2021-09-18 Parker rover session.** The contest logs show an
AZ rover at DM23XQ working a chain of SoCal stations (293–393 km) from
15:55z–20:40z. Was it rain scatter?

In [ ]:
show(os.path.join(TICKLE, "Step_4/step4_outputs/2021-09-18_sanmiguel_parker_timeline.png"),
     "KYUX corridor reflectivity vs. the 10 GHz corridor QSOs (red):")

Radar answers **yes, emphatically** that it was. Intense (60–82 dBZ) shallow morning
convection filled the corridor from 14z. The session opened inside it, and the
mid-session QSO clusters sit on local reflectivity maxima, and the final,
shortest-path contacts (20:05z, 20:40z) threaded the last spikes of a mirror
that had decayed to single-digit gate counts. After which QSOs ceased
despite 40+ hours of contest remaining. A threshold behavior captured in the
wild. As the scattering volume shrank, the workable-distance envelope shrank
with it. (The pre-15:55z high-dBZ/no-QSO period is the operator-effort term.
Propagation is necessary, not sufficient, condition for contacts.)

## 5. Geometry: the common-volume floor, and why is it important

A storm only works as a mirror if it rises into the volume visible from
*both* stations over their terrain horizons. `Step_2/common_volume.py`
computes 360 degree horizon profiles from SRTM 1-arcsec elevation (4/3-Earth
refraction) and maps the **common-volume floor**. This is the minimum 
mutually-visible altitude. It calculates this across the region for each 
site pair.

In [ ]:
print("Site-pair verdicts (from step2_outputs/summary.txt):")
p = os.path.join(TICKLE, "Step_2/step2_outputs/summary.txt")
print(open(p).read() if os.path.exists(p) else "(run common_volume.py)")

In [ ]:
# All six site-pair floor maps (the full endpoint matrix)
PAIRS = [("delmar","parker"), ("delmar","phoenix"),
         ("sanmiguel","parker"), ("sanmiguel","phoenix"),
         ("palomar","parker"), ("palomar","phoenix")]
for nk, fk in PAIRS:
    show(os.path.join(TICKLE, f"Step_2/step2_outputs/floor_{nk}__{fk}.png"),
         f"--- {nk} <-> {fk} ---")

Three results: **(1) Del Mar works** with best floor 2.0 km MSL on the
Parker path with a 10 m mast, mirror territory concentrated over the desert
slopes of the local mountains (Cuyamaca/Anza-Borrego), ~71 km from the house.
**(2) Off-axis mirrors matter** the optimizer repeatedly found the lowest
floors well off the great circle (even rescuing the mountain-blocked Palomar
cabin via Imperial Valley), matching real rain-scatter operating practice.
**(3) A hard eastern wall** near 113.5 degrees W beyond which Earth curvature 
demands impossible cell heights.

### Scope note

The geometry table and maps above cover the **full 3\u00d72 endpoint matrix**. The radar-replay results in this document, however, are for **one pair only** \u2014 San Miguel \u2194 Parker (the replay script's default endpoints, chosen as the best-labeled community pair) \u2014 on two pilot days. Labels for the other five pairs come from the bulk generation campaign in Phase 2, using the per-pair floor maps already computed. Nothing here should be read as measured results for the Del Mar or Palomar paths beyond their geometry.

### The flat-proxy vs. real-floor experiment

Until this step, the replay filtered gates with a flat 3–15 km MSL band.
Re-running benchmark day **2023-09-01** with the true San Miguel to Parker floor
map (`--floor-npz`) produced an important finding.
Recorded from the two runs, KIWA (Phoenix) evening rows:

| UTC | area40, flat 3 km proxy | area40, terrain floor |
|---|---|---|
| 21:51z | 7,601 | 54 |
| 21:57z | 6,457 | 19 |
| 22:02z | 5,447 | 65 |
| 22:13z | 2,936 | 1 |
| 22:23z | 918 | 12 |
| 22:32z | 253 | 41 |

The spectacular distant towers were mostly **below the mutual horizon**: real
weather, faithfully seen by the radar, largely invisible to the 291 km
bistatic pair whose shared volume starts 4–8 km up at that range. Reductions
run ~6× to ~3000×, smallest where echo tops reached 14–15 km. Meanwhile
overnight showers in the *western* corridor, where the floor drops below
the old 3 km cutoff, **survived and grew** (e.g. 02z: 34 to 123 gates), and
KYUX's evening storms over the low-floor western corridor registered
20,000+ qualifying gates. The geometry did not merely rescale the labels.
It relocated the day's workable mirrors. **Any label set generated without
terrain geometry would have been optimistic garbage.** High yikes!

In [ ]:
# Benchmark day under real geometry: the graded structure of a label
mt = pd.read_csv(os.path.join(TICKLE,
    "Step_4/step4_outputs/2023-09-01_sanmiguel_parker_metrics.csv"))
fig, ax = plt.subplots(figsize=(10, 4.2))
for r, c in [("KYUX", "tab:blue"), ("KIWA", "tab:red")]:
    d = mt[mt.radar == r]
    ax.semilogy(d.hour, d.area40.clip(lower=0.5), ".-", color=c, label=r)
ax.set_xlabel("UTC hour"); ax.set_ylabel("area40 (gates, log scale)")
ax.set_title("2023-09-01 under terrain geometry: 4+ decades of label dynamic range")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The log-scale view shows why Phase 2 labels will be **graded, not
binary**: the corridor's scattering volume spans four orders of magnitude in
a single day. A binary >=40 dBZ threshold saturated (93/93 positive on the
naive metrics). The graded quantity is the real target.

In [ ]:
# First training pairs: backyard GOES frames <-> radar-derived labels
# (M2 stared at the corridor all of 2023-09-01; every frame has radar truth)
frames = m.execute(
    "SELECT start_time, path FROM files WHERE parsed=1 AND product='M2' "
    "AND channel=13 AND start_time LIKE '2023-09-01%' ORDER BY start_time"
).fetchall()
radar = [(float(r.hour) * 60, float(r.max_dbz) if pd.notna(r.max_dbz) else np.nan,
          float(r.area40)) for r in mt.itertuples()]
rows, pos = [], 0
for ts, path in frames:
    t = int(ts[11:13]) * 60 + int(ts[14:16])
    near = [(d, a) for rm, d, a in radar if abs(rm - t) <= 8 and d == d]
    if near:
        dbz, a40 = max(d for d, a in near), max(a for d, a in near)
        pos += a40 >= 100
        rows.append((ts, path, dbz, a40))
pairs = pd.DataFrame(rows, columns=["goes_time", "goes_path", "max_dbz", "max_area40"])
pairs.to_csv(os.path.join(DATA, "pairs_2023-09-01.csv"), index=False)
print(f"{len(pairs)} GOES-frame/radar-label pairs; "
      f"{pos} with large mirror (area40>=100) under real geometry")
pairs.describe()

## 7. Procedure: regenerating and extending these results

All commands run inside the project venv (`source .venv/bin/activate`),
from the script's own Step directory. Paths shown are this machine's; each
script also accepts them as arguments.

**One-time setup / archive assessment (already done, rerun only if the
archive changes):**
```bash
python3 Step_1/build_manifest.py /Volumes/GOES_DRIVE --db data/goes18_manifest.db
python3 Step_1p5/estimate_pointing.py data/goes18_manifest.db      # resumable
```

**Contest ground truth (rerun when new contest years are downloaded):**
```bash
python3 parse_contest_logs.py /Users/w5nyv/TICKLE/data/arrl_10g_logs
mv contest_qsos.db /Users/w5nyv/TICKLE/data/       # delete old db first; script appends
```

**Geometry (rerun to add sites or change mast heights \u2014 edit the SITES
dict at the top of the script):**
```bash
python3 common_volume.py                 # full matrix
python3 common_volume.py --pairs delmar:parker    # one pair
```

**Radar replay for any new day/pair \u2014 the label generator.** The
criticaal rule: `--siteA/--siteB` and `--floor-npz` must describe the *same*
pair; mismatched geometry produces silent nonsense. The three configurations
used or intended so far:
```bash
# San Miguel <-> Parker (defaults; the pair used for all results above)
python3 nexrad_replay.py --date 2023-09-01 --start 0 --end 23 \\
    --radars KYUX KIWA \\
    --floor-npz /Users/w5nyv/TICKLE/Step_2/step2_outputs/floor_sanmiguel__parker.npz

# Del Mar <-> Parker (the operational home path)
python3 nexrad_replay.py --date YYYY-MM-DD --radars KYUX \\
    --siteA 32.9375,-117.2083 --siteB 33.6875,-114.0417 \\
    --floor-npz /Users/w5nyv/TICKLE/Step_2/step2_outputs/floor_delmar__parker.npz

# San Miguel <-> Phoenix (the stretch path)
python3 nexrad_replay.py --date YYYY-MM-DD --radars KIWA KYUX \\
    --siteA 32.6976,-116.9330 --siteB 33.40,-112.30 \\
    --floor-npz /Users/w5nyv/TICKLE/Step_2/step2_outputs/floor_sanmiguel__phoenix.npz
```
Volumes cache under `data/nexrad_cache/` (first pull of a full day \u2248
1\u20133 GB per radar, estimate; reruns are free). Outputs land in
`Step_4/step4_outputs/<date>_<pair>_metrics.csv` and `_<pair>_timeline.png`
(pair-keyed: runs for different pairs never overwrite each other, and every
CSV row carries `pair`, `siteA`, `siteB`, `floor_src` provenance columns); QSO overlays
appear automatically for contest dates.

**Then re-run this notebook** (Kernel \u2192 Restart & Run All) \u2014 every
figure and table above regenerates from those outputs.

### Field guide: the mistakes this project has actually made
(so they only get made once!)

**Quickstart, total amnesia case:** `cd ~/TICKLE && source .venv/bin/activate
&& jupyter notebook TICKLE_phase1.ipynb` Restart & Run All. If cells report
missing files, the procedure above regenerates them.

1. **Absolute paths, always.** `sqlite3.connect('foo.db')` on a wrong/relative
   path silently *creates an empty database*, then honestly reports
   "no such table." If you see that error where data should exist, do not
   debug the data. Hunt down the real file: `find ~/TICKLE -name '*.db' -size +1k`.
   Every 0-byte `.db` is an impostor. Delete it.
2. **The contest parser APPENDS.** Delete `data/contest_qsos.db` before
   re-running `parse_contest_logs.py`, or QSOs double-count.
3. **The pair-matching rule.** `--pair`, `--siteA/--siteB`, and `--floor-npz`
   must all describe the same site pair. A mismatch runs fine and produces
   plausible garbage. The CSV's `pair/siteA/siteB/floor_src` columns exist so
   you can audit any file's geometry after the fact. Check them when in doubt.
4. **Backslash line-continuations break sometimes.** When a
   multi-line command misbehaves, retype it as one line.
5. **Caches make reruns nearly free.** NEXRAD volumes (`data/nexrad_cache/`),
   DEM tiles (`data/dem_tiles/`), Natural Earth (`ne_cache/`) all persist.
   Deleting them costs only re-download time. They are not particularly precious.
   The precious, slow-to-recreate artifacts are the two SQLite databases and
   `pointing_segments` inside the manifest (about a 3 h census on laptop).
6. **One database, one home.** Shared data products live in `data/` only.
   A second copy of a database anywhere is a bug in waiting.
7. **Git hygiene:** `git status` before every `git add .`; `data/` stays
   ignored; the push receipt is the `old..new main -> main` line, not the
   absence of complaints.

## 6. Phase 2 design (as conclusions, not aspirations)

Phase 1's findings guide Phase 2.

1. **Metrics v2 before bulk labeling:** volume-weighted km^3 instead of gate
   counts; high-percentile dBZ instead of single-gate max (clutter/AP
   immunity); range-aware multi-radar merging. All three deficiencies were
   observed, not hypothesized.
2. **Bulk label generation:** replay every contest day 2019–2025 (plus
   ordinary days for negatives) with terrain geometry, producing graded
   labels for each site pair.
3. **Physics baseline first:** the bistatic radar equation over the floor
   maps is the champion model. Converting "visible mirror" to estimated dB.
   Any ML must beat it to justify itself.
4. **GOES features from the AWS CONUS archive** (calibrated L1b, 5-min,
   always covers the corridor), cropped to the mirror lobe the geometry
   identified (~31.5–34.5°N, 117–113.5°W) — not a naive path-line box.
   The backyard HRIT archive validates the pipeline and serves
   internet-independent live inference.
5. **Fresh ground truth:** ARRL 10 GHz & Up Leg 2 (Sept 19–21, 2026) —
   live monitoring, and recruiting an AZ-side beacon host from the operators
   this analysis identified (the DM23/DM24 regulars).

## 8. Credits

Pipeline scripts (this repo): `build_manifest.py` to `estimate_pointing.py` to
`parse_contest_logs.py` to `common_volume.py` to `nexrad_replay.py`. Heavy
inputs live outside git in `data/` (SQLite manifests, DEM tiles, NEXRAD
cache, contest log corpus. The latter re-fetchable from the ARRL log portal
via `downloader_script.py`; not redistributed here due to file size).

Data and tools gratefully used: ARRL 10 GHz & Up contest logs (portal);
NEXRAD Level II via the Unidata/AWS Open Data archive; SRTM elevation via
AWS Terrain Tiles; Natural Earth vectors; GOES-18 HRIT received on a
backyard station in Del Mar; Py-ART (Helmus & Collis 2016,
doi:10.5334/jors.119); goestools; NumPy/SciPy/pandas/matplotlib.

*Phase 1 closed with a result: the labels exist, the geometry that makes
them work well is computed, and the motivating question "can I do cloud bounce
from Del Mar?" is answered. Yes. With a mast, and a weather eye on
Anza-Borrego.*